# CellPert — End-to-End Tutorial

This notebook walks through a full CellPert use case from a clean checkout:

1. **Environment & data setup** — install dependencies and pull the LINCS / Tahoe datasets and pretrained weights from HuggingFace.
2. **Load the pretrained GINVAE model.**
3. **Build the reference biological context** from LINCS (control vs. perturb latent centroids).
4. **Predict perturbation responses** on a Tahoe query plate.
5. **Evaluate** the prediction (Pearson / Spearman / DEG-delta) on paired control–perturb samples.
All paths are relative to the repo root (`CellPert/`). After downloading the data the layout should look like:

```
CellPert/
├── src/
│   └── best_gin_vae_model_node_level.pth
├── data/
│   ├── lincs/merged_all_965_with_morgan.h5ad
│   └── minitahoe/p1_with_morgan.h5ad ... p14_with_morgan.h5ad
```

## 1. Environment & data

Run these once in a shell — *not* inside the notebook unless you want to (un)comment the `!`-prefixed lines.

```bash
conda env create -f environment.yml
conda activate info
```

Then download the model weights and datasets from HuggingFace (the repo is public, no token needed):

In [1]:
# One-time download. Everything here is skipped when the files are already in place,
# so the cell is safe to re-run.
import os
import tarfile
import zipfile

REPO_ID = 'Mike2481/CellPert'
CKPT = './src/best_gin_vae_model_node_level.pth'
LINCS_H5AD = './data/lincs/merged_all_965_with_morgan.h5ad'
TAHOE_H5AD = './data/minitahoe/p1_with_morgan.h5ad'

os.makedirs('./data', exist_ok=True)
need = [p for p in (CKPT, LINCS_H5AD, TAHOE_H5AD) if not os.path.exists(p)]
if not need:
    print('checkpoint and data already present, nothing to download')
else:
    from huggingface_hub import hf_hub_download

    if not os.path.exists(CKPT):
        hf_hub_download(repo_id=REPO_ID, filename='best_gin_vae_model_node_level.pth',
                        repo_type='dataset', local_dir='./src')

    if not os.path.exists(LINCS_H5AD):
        lincs_zip = hf_hub_download(repo_id=REPO_ID, filename='lincs.zip',
                                    repo_type='dataset', local_dir='./data')
        with zipfile.ZipFile(lincs_zip) as z:
            z.extractall('./data/')

    if not os.path.exists(TAHOE_H5AD):
        tahoe_tar = hf_hub_download(repo_id=REPO_ID, filename='minitahoe.tar.gz',
                                    repo_type='dataset', local_dir='./data')
        with tarfile.open(tahoe_tar, 'r:gz') as t:
            t.extractall('./data/')
    print('downloaded:', ', '.join(need))


checkpoint and data already present, nothing to download


## 2. Load the pretrained GINVAE

`GINVAE` defaults: `input_dim=1, hidden_dim=300, latent_dim=100, num_layers=2`. Match these to the checkpoint.

In [2]:
import sys
sys.path.insert(0, './src')

import torch
from model import GINVAE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GINVAE(input_dim=1, hidden_dim=300, latent_dim=100, num_layers=2).to(device)
model.load_state_dict(torch.load('./src/best_gin_vae_model_node_level.pth', map_location=device))
model.eval();

## 3. Build the reference biological context

CellPert transfers a perturbation effect from a *reference* dataset (LINCS) to a *query* dataset (Tahoe). The reference contributes two centroids in latent space:

$$z^{\text{perturb}}_{\text{query}} = z^{\text{ctrl}}_{\text{query}} + (\bar z^{\text{perturb}}_{\text{ref}} - \bar z^{\text{ctrl}}_{\text{ref}})$$

The first call processes LINCS and caches `latent_ctrl_ref` / `latent_ptrb_ref` to disk; subsequent calls are instant.

In [3]:
from dataset import ChunkedGeneGraphDataset
import os

os.makedirs('./output', exist_ok=True)

lincs_path = './data/lincs/merged_all_965_with_morgan.h5ad'
lincs_dataset = ChunkedGeneGraphDataset(
    h5ad_paths=[lincs_path], split='train',
    chunk_size=10000, auto_build_graph=True,
    species=9606, required_score=700,
)

latent_ctrl_ref, latent_ptrb_ref, info = model.process_reference_dataset(
    x_ref_dataset=lincs_dataset,
    save_path='./output/reference_latents.pkl',
    force_reprocess=False,
)
print('ctrl latent shape :', latent_ctrl_ref.shape)
print('perturb latent shape:', latent_ptrb_ref.shape)

Found 16 chunks with 157138 total samples
Loading existing processed data from ./output/reference_latents.pkl
Loaded reference data metadata from ./output/reference_latents.pkl
Total samples available: 157138
Loading data from chunks...


Loading chunks:   0%|          | 0/16 [00:00<?, ?it/s]

Loading chunks:   0%|          | 0/16 [00:00<?, ?it/s, file=chunk_0.pkl]

Loading chunks:   0%|          | 0/16 [00:00<?, ?it/s, file=chunk_1.pkl]

Loading chunks:   0%|          | 0/16 [00:00<?, ?it/s, file=chunk_2.pkl]

...

Processing perturb samples:  85%|████████▌ | 67054/78569 [00:01<00:00, 51157.47it/s]

Processing perturb samples:  96%|█████████▌| 75512/78569 [00:01<00:00, 58441.08it/s]

Processing perturb samples: 100%|██████████| 78569/78569 [00:01<00:00, 64350.45it/s]

Computing mean representations...
References computed successfully.
Control ref shape: torch.Size([1, 100]), norm: 2.2677
perturb ref shape: torch.Size([1, 100]), norm: 2.2584
Biological context norm: 0.0157


ctrl latent shape : torch.Size([1, 100])
perturb latent shape: torch.Size([1, 100])


## 4. Predict perturbation responses on a query plate

In [4]:
from torch_geometric.data import DataLoader

tahoe_path = './data/minitahoe/p1_with_morgan.h5ad'
tahoe_dataset = ChunkedGeneGraphDataset(
    h5ad_paths=[tahoe_path], split='test',
    chunk_size=10000, auto_build_graph=True,
    species=9606, required_score=700,
)

loader = DataLoader(tahoe_dataset, batch_size=64, shuffle=False)
batch = next(iter(loader)).to(device)
predicted_exp, mask = model.predict(batch, latent_ctrl_ref, latent_ptrb_ref)
print('predicted expression shape:', predicted_exp.shape)

Found 4 chunks with 39546 total samples


predicted expression shape: torch.Size([64, 965, 1])


## 5. Evaluate on paired control–perturb samples

Re-uses the same evaluation pipeline that `src/main.py` runs in `--test_flag` mode:

* Filters Tahoe by `condition ∈ {control, perturb}`.
* Pairs them on `(celltype, main_ptrb, sub_ptrb)`.
* Computes per-sample Pearson / Spearman on absolute expression and on the DEG delta `pred − ctrl`.

Two equivalent ways to run it:

In [5]:
# Option A — call the helper directly from main.py
import argparse
from main import evaluate_perturbation_prediction

args = argparse.Namespace(
    device=device, batch_size=64,
    output_dir='./output', test_dataset_id=1,
)
results = evaluate_perturbation_prediction(
    model, tahoe_dataset, latent_ctrl_ref, latent_ptrb_ref,
    args, max_test_pairs=200, compute_deg=True,
)
print('absolute :', results['absolute_metrics'])
print('DEG delta:', results['deg_metrics'])

Filtering test dataset by condition...
Filtering dataset for condition: control


Filtering control samples:   0%|          | 0/39546 [00:00<?, ?it/s]

Filtering control samples:  25%|██▌       | 10001/39546 [00:00<00:01, 19728.23it/s]

Filtering control samples:  51%|█████     | 20001/39546 [00:01<00:01, 18593.10it/s]

Filtering control samples:  76%|███████▌  | 30001/39546 [00:01<00:00, 17566.62it/s]

...

Predicting perturbations:  50%|█████     | 2/4 [00:00<00:00, 11.85it/s]

Predicting perturbations: 100%|██████████| 4/4 [00:00<00:00, 15.30it/s]

Processing prediction results...
Processed 200 prediction pairs



Perturbation Prediction Metrics (Control → Predicted Stimulated vs Ground Truth Stimulated):
MSE                  39.903273
MAE                   6.304446
R2                 -371.725347
Pearson               0.348541
Spearman              0.153887
CosineSimilarity      0.407824
JS_Divergence         0.303515
SSIM                  0.011863
TopK_Overlap          0.008200
total_samples       200.000000
dtype: float64

Computing DEG (delta difference) metrics...

DEG Prediction Metrics (Predicted Delta vs Ground Truth Delta):
MSE                   39.904005
MAE                    6.304653
R2                 -1035.438793
Pearson                0.176272
Spearman               0.188740
CosineSimilarity       0.018459
JS_Divergence          0.004992
SSIM                  -0.004161
TopK_Overlap           0.058450
total_samples        200.000000
dtype: float64

✓ Results appended to: ./output/all_plates_results.csv
✓ Predictions saved to: ./output/predictions/plate_1_predictions.pkl


✓ Single plate results saved to: ./output/plate_1_results.csv
absolute : {'MSE': 39.90327272415161, 'MAE': 6.304446341991425, 'R2': -371.7253472518921, 'Pearson': 0.3485410862416029, 'Spearman': 0.15388651574961842, 'CosineSimilarity': 0.4078244514763355, 'JS_Divergence': 0.30351463086903097, 'SSIM': 0.011863052063272334, 'TopK_Overlap': 0.0082, 'total_samples': 200}
DEG delta: {'MSE': 39.90400453567505, 'MAE': 6.3046529006958005, 'R2': -1035.438793182373, 'Pearson': 0.17627162333810703, 'Spearman': 0.188739739524608, 'CosineSimilarity': 0.018459166332613675, 'JS_Divergence': 0.0049916924626450055, 'SSIM': -0.0041612324023731165, 'TopK_Overlap': 0.05845000000000001, 'total_samples': 200}


In [6]:
# Option B — full sweep from the shell (one CSV row per plate)
# bash src/run_all.sh
# After it finishes, results land in:
#   output/all_plates_results.csv
#   output/predictions/plate_*_predictions.pkl